In [0]:
import os
import re

def get_next_versioned_dir(src_dir: str) -> str:
    """
    Given a source directory path, return the next versioned path.
    Examples:
      /Volumes/.../dbo        → /Volumes/.../dbo_I
      /Volumes/.../dbo_I      → /Volumes/.../dbo_II
      /Volumes/.../dbo_II     → /Volumes/.../dbo_III
      /Volumes/.../dbo_III    → /Volumes/.../dbo_IV
    """
    vol_root  = os.path.dirname(src_dir)
    subfolder = os.path.basename(src_dir)

    # Match trailing roman numeral suffix added by this script
    match = re.match(r"^(.+?)(_[IVX]+)?$", subfolder)
    base   = match.group(1)
    suffix = match.group(2)[1:] if match.group(2) else ""  # strip leading _

    # Roman numeral increment map
    roman_sequence = ["", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]
    
    if suffix in roman_sequence:
        next_suffix = roman_sequence[roman_sequence.index(suffix) + 1]
    else:
        raise ValueError(f"Unrecognized suffix '{suffix}' on folder '{subfolder}'. Extend roman_sequence if needed.")

    next_folder = f"{base}_{next_suffix}"
    return os.path.join(vol_root, next_folder)


def find_available_dst(src_path: str) -> str:
    """
    Walk up the versioned folder chain until we find one that doesn't
    already contain the target file, then return that destination path.
    """
    src_dir      = os.path.dirname(src_path)
    filename     = os.path.basename(src_path)
    current_dir  = src_dir

    while True:
        candidate_dir  = get_next_versioned_dir(current_dir)
        candidate_path = os.path.join(candidate_dir, filename)
        if not os.path.exists(candidate_path):
            return candidate_path
        # File already exists in this version folder → try next
        current_dir = candidate_dir


# ── 1. Read the log table ────────────────────────────────────────────────────
dbutils.widgets.text("source_catalog", "catalog")
dbutils.widgets.text("source_schema", "schema")
dbutils.widgets.text("log_table", "switch_run_log")

source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")
log_table      = dbutils.widgets.get("log_table")

df = spark.table(f"dbe_dbx_internships.log.run_details")

TARGET_ERROR_FRAGMENT = "Unable to access the notebook"

failed_rows = (
    df.filter(
        (df.run_status == "FAILED") &
        (df.run_error.contains(TARGET_ERROR_FRAGMENT))
    )
    .select("input_file_path", "input_file_relative_path")
    .collect()
)

print(f"Found {len(failed_rows)} FAILED file(s) matching 'Unable to access the notebook'.")
# ── 2. Re-upload each failed file to the next versioned folder ───────────────
results = []

for row in failed_rows:
    src_path = row["input_file_path"]
    rel_path = row["input_file_relative_path"]

    try:
        dst_path = find_available_dst(src_path)
        dst_dir  = os.path.dirname(dst_path)

        with open(src_path, "rb") as f:
            content = f.read()

        os.makedirs(dst_dir, exist_ok=True)

        with open(dst_path, "wb") as f:
            f.write(content)

        results.append({"file": rel_path, "dst": dst_path, "status": "OK"})
        print(f"  ✓  {rel_path}  →  {dst_path}")

    except Exception as e:
        results.append({"file": rel_path, "dst": dst_path if 'dst_path' in dir() else "N/A", "status": f"ERROR: {e}"})
        print(f"  ✗  {rel_path}  →  ERROR: {e}")

# ── 3. Summary ───────────────────────────────────────────────────────────────
ok    = sum(1 for r in results if r["status"] == "OK")
error = sum(1 for r in results if r["status"] != "OK")
print(f"\nDone — {ok} re-uploaded, {error} error(s).")